In [1]:
import pandas as pd
import altair as alt

# Seminar: the Long-Run Prices Dataset

A notebook using the LRPD, described [here](https://cep.lse.ac.uk/pubs/download/occasional/op055.pdf).

</br></br></br></br>


Let's take a look at the prices dataset first

In [2]:
prices_df = pd.read_parquet('https://autocpi-public.s3.eu-west-2.amazonaws.com/lrpd/db_prices.parquet')
prices_df.describe()

,quote_date,shop_code,item_id_raw,region,price,item_id
count,4.836896e+07,4.836896e+07,4.836897e+07,4.836896e+07,4.836896e+07,4.836897e+07
mean,2.007762e+05,4.771270e+02,3.880409e+05,6.679112e+00,4.960397e+01,3.883983e+05
std,1.060226e+03,1.531775e+03,1.467557e+05,3.407499e+00,2.081768e+02,1.466723e+05
min,1.988020e+05,1.000000e+00,2.101010e+05,1.000000e+00,1.000000e-02,2.101010e+05
25%,1.998110e+05,3.900000e+01,2.129170e+05,3.000000e+00,1.490000e+00,2.129180e+05
50%,2.008050e+05,8.800000e+01,4.301280e+05,7.000000e+00,4.850000e+00,4.301320e+05
75%,2.017070e+05,8.020000e+02,5.104060e+05,9.000000e+00,1.999000e+01,5.104070e+05
max,2.025100e+05,2.007100e+04,6.404060e+05,1.300000e+01,4.400000e+04,6.404060e+05


We've got 48 million observations and a dates from 1982 to 2025.
</br></br></br></br>


We also need the items data to understand what each product is.

In [3]:
items_df = pd.read_parquet('https://autocpi-public.s3.eu-west-2.amazonaws.com/lrpd/db_item.parquet')
items_df.head()

,item_id,description,date_quote_s,date_quote_e,n_obs
0,210101,LARGE LOAF-WHITE-SLICED-800G,198802,200401,36039
1,210102,LARGE LOAF-WHITE-UNSLICED-800G,198802,202510,56917
2,210105,LARGE WHOLEMEAL LOAF-UNSLICED,198802,200301,27161
3,210106,SIX BREAD ROLLS-WHITE/BROWN,198802,202510,67469
4,210107,"BROWN LOAF,400G,SLICED-GRAN",198903,200401,29361


</br></br></br>
# Simple chart: the price of milk 

Let's chart the price of milk. First we need to work out what the `item_id` is. Let's look in the items_df to find out.
</br></br>

In [7]:
items_df[items_df['description'].str.contains('tea', case=False)]

,item_id,description,date_quote_s,date_quote_e,n_obs
41,210318,FROZN CAKE/GATEAU NO ICE-CREAM,199502,200601,18229
50,210406,HOME KLD BEEF-RUMP/POPES STEAK,198802,202510,117178
51,210407,HOME KILLED BEEF-STEWING STEAK,198802,200201,47235
54,210415,H-KILLED BEEF BRAISING STEAK,200202,201501,33209
61,210506,HK LAMB LOIN CHOP/STEAK PER KG,201502,202510,38296
69,210707,HOME KILLED PORK CHOP/STEAK,202402,202510,5531
83,210911,FRESH TURKEY STEAKS PER KG,200402,202001,10135
91,211007,CANNED MEAT-STEWED STEAK,198802,202510,57641
176,211901,TEA BAGS PKT OF 80 (230G-250G),198802,202510,72190
177,211902,TEA-LOOSE-125G,198802,200201,29951


Let's go for `211710` - `MILK SEMI-PER 2 PINTS/1.136 L`. It's got a long timeseries (1992-202510) and is probably representative.

</br></br></br></br>

First, let's just plot the mean, median and median price of milk over time.

In [9]:
tea_prices = prices_df.query("item_id == 220305")
avg_tea_prices = tea_prices.groupby('quote_date').agg({'price': ['mean', 'median']}).reset_index()
    
avg_tea_prices.columns = ['date', 'Mean', 'Median']
avg_tea_prices

,date,Mean,Median
0,198802.0,0.239290,0.24
1,198803.0,0.245359,0.25
2,198804.0,0.247423,0.24
3,198805.0,0.247170,0.25
4,198806.0,0.250617,0.25
...,...,...,...
424,202309.0,1.941107,2.00
425,202310.0,1.947235,2.00
426,202311.0,1.963711,2.00
427,202312.0,1.964980,2.00


To use it in Altair/Vega-lite, we just have to melt it from wide to long.

In [11]:
tea_prices

,quote_date,shop_code,item_id_raw,region,price,indicator_box,item_id
15095684,199804.0,104.0,220305,8.0,0.49,,220305
15095685,201409.0,92.0,220305,6.0,1.75,,220305
15095686,199101.0,61.0,220305,2.0,0.30,,220305
15095687,199303.0,151.0,220305,7.0,0.25,,220305
15095688,201002.0,15.0,220305,8.0,1.10,,220305
...,...,...,...,...,...,...,...
15211426,201007.0,8.0,220305,8.0,0.82,,220305
15211427,201701.0,39.0,220305,3.0,1.10,,220305
15211428,200811.0,11.0,220305,3.0,0.55,,220305
15211429,201009.0,27.0,220305,5.0,1.00,,220305


In [12]:
avg_tea_prices_melted = avg_tea_prices.melt(id_vars=['date'], 
                                                   var_name='price_type',
                                                   value_name='price')

# and we can also just format the date nicely, so altair treats them properly
avg_tea_prices_melted['date'] = pd.to_datetime(avg_tea_prices_melted['date'], format='%Y%m')

avg_tea_prices_melted

,date,price_type,price
0,1988-02-01,Mean,0.239290
1,1988-03-01,Mean,0.245359
2,1988-04-01,Mean,0.247423
3,1988-05-01,Mean,0.247170
4,1988-06-01,Mean,0.250617
...,...,...,...
853,2023-09-01,Median,2.000000
854,2023-10-01,Median,2.000000
855,2023-11-01,Median,2.000000
856,2023-12-01,Median,2.000000


In [14]:
alt.Chart(avg_tea_prices_melted).mark_line().encode(
    x=alt.X('date:T', title=''),
    y=alt.Y('price:Q', title='Price (GBP)'),
    color='price_type:N'
).properties(
    title={
        "text": "Tea Prices",
        "subtitle": ["Mean and Median Prices for Tea", "Source: ONS microdata via Davies (2021)"],
        "fontSize": 16
    }
)

alt.Chart(...)